# 이터널 리턴 콘텐츠 및 메타 데이터 전처리
> **목적**: 캐릭터 번호로 기록된 인게임 데이터를 이름 및 역할군으로 매핑하고, 밸런스 분석을 위한 성능 지표를 집계합니다.

### 주요 작업 내용:
1. **데이터 매핑**: 캐릭터 ID를 실제 이름(`character_map`) 및 역할군(Dealer, Tanker 등)으로 치환
2. **성능 지표 집계**: 캐릭터별 평균 순위(Rank), 킬(Kill), 생존 시간 등을 그룹화하여 산출
3. **경로 데이터 정제**: 유저의 이동 경로(Sankey 분석용)를 'From-To' 구조로 변환
4. **결측치 처리**: 업데이트로 인한 신규 캐릭터 번호 및 예외 데이터 정제

# Content Data Preprocessing

## STEP 0. 환경 설정
- 라이브러리 import
- 기본 환경 설정

In [ ]:
# STEP 0. 환경 설정

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

## STEP 1. 데이터 로드
- 원본 데이터 불러오기
- 기본 구조 확인

In [ ]:
# STEP 1. 데이터 로드

df = pd.read_csv('data/raw/game_data.csv')

df.head()

## STEP 2. 캐릭터 메타 정의
- 캐릭터 역할(Role) 매핑
- 메타 정보 생성

In [ ]:
# STEP 2. 캐릭터 메타 정의

character_role = {
    '아야': '딜러',
    '매그너스': '탱커',
    '하트': '서포터'
}

df['role'] = df['character'].map(character_role)

## STEP 3. 데이터 정제
- 결측치 제거
- 비정상 데이터 필터링

In [ ]:
# STEP 3. 데이터 정제

df = df.dropna(subset=['character', 'rank'])

df = df[df['playTime'] > 0]

## STEP 4. 파생 변수 생성
- 무기 타입 분류
- 전투 스타일 정의
- 조기 사망 여부 생성

In [ ]:
# STEP 4. 파생 변수 생성

# 무기 타입
df['weapon_type'] = df['weapon'].str.split('_').str[0]

# 전투 스타일
df['combat_style'] = np.where(df['kill'] >= 5, 'Aggressive', 'Passive')

# 조기 사망 여부
df['early_death'] = (df['playTime'] < 300).astype(int)

## STEP 5. 경로 데이터 변환
- 이동 경로 리스트 변환
- From-To 구조 생성 (Sankey용)

In [ ]:
# STEP 5. 경로 데이터 변환

# 예: "A > B > C"
df['path_list'] = df['route'].str.split(' > ')

# from-to 구조 생성
edges = []

for path in df['path_list'].dropna():
    for i in range(len(path)-1):
        edges.append((path[i], path[i+1]))

path_df = pd.DataFrame(edges, columns=['from', 'to'])

## STEP 6. 캐릭터 성능 집계
- 캐릭터별 성능 지표 계산
- 평균 랭크 / 킬 / 생존 등

In [ ]:
# STEP 6. 캐릭터 성능 집계

char_stats = df.groupby('character').agg({
    'rank': 'mean',
    'kill': 'mean',
    'playTime': 'mean',
    'early_death': 'mean'
}).reset_index()

## STEP 7. 데이터 저장
- 콘텐츠 분석용 데이터 저장
- 경로 데이터 저장

In [ ]:
# STEP 7. 데이터 저장

char_stats.to_csv('data/processed/content_final.csv', index=False)
path_df.to_csv('data/processed/path_data.csv', index=False)